### 1 - Data Ingesion & Integration

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when

In [2]:
#Dynamic Path Detection so it runs on all user environments
BASE_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(BASE_DIR, "data")

In [3]:
#Spark Initialization
spark = SparkSession.builder \
    .appName("BODS_Predictive_Engine") \
    .config("spark.hadoop.fs.defaultFS", "file:///") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/02 11:20:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
# Load files
df_cat = spark.read.csv(os.path.join(DATA_DIR, "overall_data_catalogue.csv"), header=True, inferSchema=True)
df_comp = spark.read.csv(os.path.join(DATA_DIR, "overall_compliance_report.csv"), header=True, inferSchema=True)

# Surgical Join: Joining on Service Code (Catalogue) and Service Number (Compliance) <Multi Catalogue Ingestion>
df_unified = df_cat.join(
    df_comp, 
    df_cat["Service Code"] == df_comp["Registration:Service Number"], 
    how="right"
)

print(f"Row count after surgical join: {df_unified.count()}")

[Stage 21:>                                                         (0 + 2) / 2]

Row count after surgical join: 103437


### 2 - Data Cleaning

In [8]:
# 1. Drop rows where critical compliance data is missing
df_cleaned = df_unified.fillna({'Service Code': 'UNKNOWN', 'Status': 'UNKNOWN'})

# 2. Impute missing numeric values (replace NULLs with 0)
df_cleaned = df_cleaned.fillna(0, subset=["% AVL to Timetables feed matching score"])

# 3. Final validation of record count
final_count = df_cleaned.count()
print(f"### **Final Cleaned Record Count: {final_count}**")

[Stage 30:===========================================>              (3 + 1) / 4]

### **Final Cleaned Record Count: 103437**
